In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import math
import joblib
import os

df = pd.read_csv('ml-1m/ratings.dat',
                 sep='::',
                 engine='python',
                 names=['user_id', 'product_id', 'rating', 'timestamp'])

df = df[['user_id', 'product_id', 'rating']]
df['user_id'] = pd.Categorical(df['user_id']).codes
df['product_id'] = pd.Categorical(df['product_id']).codes

NUM_USERS = df['user_id'].nunique()
NUM_PRODUCTS = df['product_id'].nunique()

print(f"Users: {NUM_USERS}, Products: {NUM_PRODUCTS}, Ratings: {len(df)}")
print(df['rating'].describe())


Users: 6040, Products: 3706, Ratings: 1000209
count    1.000209e+06
mean     3.581564e+00
std      1.117102e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64


In [2]:
R = csr_matrix(
    (df['rating'], (df['user_id'], df['product_id'])),
    shape=(NUM_USERS, NUM_PRODUCTS)
)
print("Matrix shape:", R.shape)
print("Non-zero entries:", R.nnz)

Matrix shape: (6040, 3706)
Non-zero entries: 1000209


In [3]:
K = 50

# Dense matrix
R_dense = R.toarray().astype(np.float64)

# only rated items mean (zeros exclude)
rated_counts = (R_dense != 0).sum(axis=1)
rated_counts[rated_counts == 0] = 1  # division by zero avoid
user_ratings_mean = R_dense.sum(axis=1) / rated_counts

# Subtract mean only from rated positions
R_demeaned = R_dense.copy()
R_demeaned[R_dense != 0] -= np.repeat(user_ratings_mean, (R_dense != 0).sum(axis=1))

# SVD
U, sigma, Vt = svds(csr_matrix(R_demeaned), k=K)
idx = np.argsort(sigma)[::-1]
sigma = sigma[idx]
U = U[:, idx]
Vt = Vt[idx, :]
Sigma = np.diag(sigma)

# Add mean back
R_predicted = np.dot(np.dot(U, Sigma), Vt) + user_ratings_mean.reshape(-1, 1)
R_predicted = np.clip(R_predicted, 1, 5)

print("U shape:", U.shape)
print("Sigma shape:", Sigma.shape)
print("Vt shape:", Vt.shape)
print("R_predicted mean:", R_predicted.mean())   # ~3.58 hona chahiye
print("R_predicted std:", R_predicted.std())

U shape: (6040, 50)
Sigma shape: (50, 50)
Vt shape: (50, 3706)
R_predicted mean: 3.702616189932703
R_predicted std: 0.4422018428485473


In [4]:
def recommend(user_id, n=5):
    user_scores = R_predicted[user_id].copy()
    rated_products = df[df['user_id'] == user_id]['product_id'].values
    user_scores[rated_products] = -999
    top_n = np.argsort(user_scores)[::-1][:n]
    return top_n

print("Top 5 for user 0:", recommend(0))
print("Top 5 for user 42:", recommend(42))

Top 5 for user 0: [346 106 466 354 576]
Top 5 for user 42: [   0 2748 2162 2511 1449]


In [5]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

R_train = csr_matrix(
    (train_df['rating'], (train_df['user_id'], train_df['product_id'])),
    shape=(NUM_USERS, NUM_PRODUCTS)
)

# Mean center train matrix
R_train_dense = R_train.toarray().astype(np.float64)
train_counts = (R_train_dense != 0).sum(axis=1)
train_counts[train_counts == 0] = 1
train_mean = R_train_dense.sum(axis=1) / train_counts

R_train_demeaned = R_train_dense.copy()
R_train_demeaned[R_train_dense != 0] -= np.repeat(train_mean, (R_train_dense != 0).sum(axis=1))

U2, sigma2, Vt2 = svds(csr_matrix(R_train_demeaned), k=50)
idx2 = np.argsort(sigma2)[::-1]
sigma2 = sigma2[idx2]
U2 = U2[:, idx2]
Vt2 = Vt2[idx2, :]

R_pred2 = np.dot(np.dot(U2, np.diag(sigma2)), Vt2) + train_mean.reshape(-1, 1)
R_pred2 = np.clip(R_pred2, 1, 5)

actual = test_df['rating'].values
predicted = [R_pred2[int(row.user_id), int(row.product_id)] for _, row in test_df.iterrows()]

rmse = math.sqrt(mean_squared_error(actual, predicted))
print(f"RMSE: {rmse:.4f}")

RMSE: 0.9682


In [6]:
def precision_recall_ndcg_at_k(R_predicted, R_actual, k=10):
    precisions, recalls, ndcgs = [], [], []
    for user in range(R_predicted.shape[0]):
        actual_liked = set(np.where(R_actual[user] >= 4)[0])
        if not actual_liked:
            continue
        top_k_pred = np.argsort(R_predicted[user])[::-1][:k]
        hits = [1 if p in actual_liked else 0 for p in top_k_pred]
        precision = sum(hits) / k
        recall = sum(hits) / len(actual_liked)
        dcg = sum([h / np.log2(i + 2) for i, h in enumerate(hits)])
        ideal_hits = [1] * min(len(actual_liked), k)
        idcg = sum([h / np.log2(i + 2) for i, h in enumerate(ideal_hits)])
        ndcg = dcg / idcg if idcg > 0 else 0
        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)
    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

R_actual_full = np.zeros((NUM_USERS, NUM_PRODUCTS))
for _, row in df.iterrows():
    R_actual_full[int(row.user_id)][int(row.product_id)] = row.rating

p, r, n = precision_recall_ndcg_at_k(R_predicted, R_actual_full, k=10)

print(f"\n===== FINAL METRICS =====")
print(f"RMSE:         {rmse:.4f}")
print(f"Precision@10: {p:.4f}")
print(f"Recall@10:    {r:.4f}")
print(f"NDCG@10:      {n:.4f}")


===== FINAL METRICS =====
RMSE:         0.9682
Precision@10: 0.7266
Recall@10:    0.1430
NDCG@10:      0.7787


In [7]:
os.makedirs("model", exist_ok=True)
joblib.dump(U,                 "model/U.pkl")
joblib.dump(sigma,             "model/sigma.pkl")
joblib.dump(Vt,                "model/Vt.pkl")
joblib.dump(df,                "model/ratings_df.csv")
joblib.dump(user_ratings_mean, "model/user_ratings_mean.pkl")
print("\nModel saved OK")
print(f"U: {U.shape}, Vt: {Vt.shape}")


Model saved OK
U: (6040, 50), Vt: (50, 3706)
